# 2 - Fine-tuning of EoMT-S on LoveDA

Using the same pipeline as the paper do in `main.py fit` with the config `configs/dinov2/loveda/semantic/eomt_small_640.yaml`].

It starts with COCO weights (`--model.ckpt_path`) but the classification head get reneizializated (`--model.load_ckpt_class_head False`)
Using batch 4 x accumulation 4 = bath size of 16 as in the paper
Mask annealing is present but scaled to the training

In [ ]:
from pathlib import Path

DATA_DIR = Path("data/loveda")
CKPT_PATH = Path("checkpoints/coco_panoptic_eomt_small_640_2x.bin")
CONFIG = Path("configs/dinov2/loveda/semantic/eomt_small_640.yaml")

assert DATA_DIR.exists() and CKPT_PATH.exists() and CONFIG.exists(), "Need 0_Setup.ipynb"

## 1. Check of steps

In [ ]:
import yaml

from datasets.loveda_semantic import LoveDASemantic

cfg = yaml.safe_load(CONFIG.read_text())
batch_size = cfg["data"]["init_args"]["batch_size"]
accumulate = cfg["trainer"]["accumulate_grad_batches"]
max_epochs = cfg["trainer"]["max_epochs"]

n_train = len(LoveDASemantic(path=str(DATA_DIR), batch_size=batch_size, num_workers=0).setup().train_dataset)
steps_per_epoch = n_train // batch_size // accumulate
total_steps = steps_per_epoch * max_epochs

print(f"Immagini train        : {n_train}")
print(f"Optimizer step/epoca  : {steps_per_epoch}")
print(f"Step totali           : {total_steps}")
print(f"Annealing end (config): {cfg['model']['init_args']['attn_mask_annealing_end_steps']}")
assert max(cfg["model"]["init_args"]["attn_mask_annealing_end_steps"]) < total_steps, \
    "L'annealing non finirebbe entro il training: riscala gli step nella config!"

## 2. Fine-tuning from COCO checkpoint

It also prints the command to execute in the cmd, useless as its a notebook but i thought it was cool/nice optional()

In [ ]:
import os, subprocess, sys

def run_training(extra_args, run_name):
    cmd = [
        sys.executable, "main.py", "fit",
        "-c", str(CONFIG),
        "--compile_disabled",
        "--data.path", str(DATA_DIR),
        "--trainer.logger.init_args.name", run_name,
        *extra_args,
    ]
    print(" ".join(cmd), "\n")
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
        env={**os.environ, "PYTHONUTF8": "1"},
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    assert proc.returncode == 0, f"Training fallito (exit code {proc.returncode})"

run_training(
    extra_args=[
        "--model.ckpt_path", str(CKPT_PATH),
        "--model.load_ckpt_class_head", "False",
    ],
    run_name="loveda_ft_da_coco",
)

## 3. Ablation test, fine-tuning using only DINOv2 weights

Same stuff butb without COCO checkpoint.  
Should let us understand how much of the gains comes from the pre-training on COCO and how much on DINOv2.

In [ ]:
RUN_ABLATION = True

if RUN_ABLATION:
    run_training(extra_args=[], run_name="loveda_ft_da_dinov2")